# Bakeries Data Layer – Transformation & Preprocessing

# Objective 
    Prepare a unified, validated bakery dataset ready for integration into the platform database.

# Libraries

In [35]:
# Import Libraries
import pandas as pd
import geopandas as gpd # to work with geospatial data
import osmnx as ox # to fetch data from OpenStreetMap
from shapely.geometry import Point
import json
import requests
import numpy as np

In [36]:
# Set pandas option to avoid future warnings
pd.set_option('future.no_silent_downcasting', True)

# 1. Data Extraction – OpenStreetMap (Primary Source)

 ##  Bakery Definition

Establishments primarily selling bread and pastries.

 ## Eligible OSM Tags

 - shop=bakery

 - shop=pastry

- bakery=yes

In [37]:
ox.settings.log_console = True
ox.settings.use_cache = True


osm_tags = {
"shop": ["bakery", "pastry"],
"bakery": "yes"
}


bakeries_raw = ox.features_from_place(
"Berlin, Germany",
tags=osm_tags
).reset_index()

#len(bakeries_raw)
bakeries_raw.head()

,element,id,geometry,addr:city,addr:country,addr:housenumber,addr:postcode,addr:street,addr:suburb,shop,...,room,roof:levels,office,building:part,covered,nohousenumber,coffee,location,shop_1,description:de
0,node,58489996,POINT (13.40829 52.51035),Berlin,DE,77,10179,Alte Jakobstraße,Mitte,bakery,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,node,253999302,POINT (13.35618 52.48628),Berlin,DE,22,10827,Hauptstraße,Schöneberg,bakery,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,node,254333065,POINT (13.32023 52.49613),NaN,NaN,NaN,NaN,NaN,NaN,bakery,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,node,254992626,POINT (13.26299 52.48796),NaN,NaN,NaN,NaN,NaN,NaN,bakery,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,node,258050855,POINT (13.29729 52.50696),Berlin,DE,80,10627,Kantstraße,Charlottenburg,bakery,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [38]:
bakeries_raw.columns.tolist()

['element',
 'id',
 'geometry',
 'addr:city',
 'addr:country',
 'addr:housenumber',
 'addr:postcode',
 'addr:street',
 'addr:suburb',
 'shop',
 'bakehouse',
 'brand',
 'brand:wikidata',
 'brand:wikipedia',
 'check_date:opening_hours',
 'contact:website',
 'indoor_seating',
 'name',
 'opening_hours',
 'outdoor_seating',
 'wheelchair',
 'toilets:wheelchair',
 'contact:phone',
 'smoking',
 'operator',
 'name:de',
 'amenity',
 'created_by',
 'description',
 'website',
 'contact:facebook',
 'contact:instagram',
 'diet:vegetarian',
 'email',
 'internet_access',
 'internet_access:fee',
 'phone',
 'internet_access:ssid',
 'source',
 'cuisine',
 'dog',
 'access:covid19',
 'delivery:covid19',
 'opening_hours:signed',
 'payment:credit_cards',
 'payment:debit_cards',
 'takeaway',
 'check_date',
 'level',
 'payment:cash',
 'payment:coins',
 'payment:notes',
 'note',
 'air_conditioning',
 'contact:email',
 'delivery',
 'delivery:partner',
 'nobrand',
 'organic',
 'payment:american_express',
 'paymen

In [39]:
# Check for common amenity/accessibility tags in the raw data
tags_to_check = ['wheelchair', 'outdoor_seating', 'indoor_seating', 'takeaway', 'delivery', 'self_service']
available_tags = [col for col in bakeries_raw.columns if col in tags_to_check]

print("Available OSM tags for enrichment:")
print(available_tags)

# See a sample of the data for these tags
if available_tags:
    print(bakeries_raw[available_tags].dropna(how='all').head())
else:
    print("No matching tags found. You might need to check bakeries_raw.columns for similar names.")

Available OSM tags for enrichment:
['indoor_seating', 'outdoor_seating', 'wheelchair', 'takeaway', 'delivery', 'self_service']
  indoor_seating outdoor_seating wheelchair takeaway delivery self_service
1            yes             yes        yes      NaN      NaN          NaN
2            yes             yes         no      NaN      NaN          NaN
3            NaN             NaN        yes      NaN      NaN          NaN
5            yes             yes        yes      NaN      NaN          NaN
6            yes             yes        yes      NaN      NaN          NaN


# 2. Initial Inspection

In [40]:
bakeries_raw.head()
#Validate tagging strategy and detect overlaps with cafes or restaurants.
bakeries_raw[["name", "shop", "bakery", "amenity"]].sample(10)

,name,shop,bakery,amenity
1150,The Sanctuary,bakery,NaN,NaN
88,Ditsch,bakery,NaN,NaN
1118,Freunde Back,bakery,NaN,NaN
1114,Märkisch Edel,bakery,NaN,NaN
490,Çevik,bakery,NaN,NaN
979,Backwerk,bakery,NaN,NaN
1238,Happy Back,bakery,NaN,NaN
507,Steinecke,bakery,NaN,NaN
356,Backshop,bakery,NaN,NaN
1279,KoJaKe,pastry,NaN,NaN


Inspection of OSM bakery tags confirms that Berlin bakeries are predominantly classified as  shop=bakery, with a smaller but relevant subset tagged as shop=pastry (e.g., Konditoreien and pastry-focused shops). Both tags are therefore included in the unified Bakeries dataset.

# 3. Column Cleaning & Standardization
## Rules

- All columns lowercase

- Snake_case naming

- Unified address fields#

In [41]:
bakeries_raw.columns = [
col.lower().replace(":", "_") for col in bakeries_raw.columns
]


rename_map = {
"addr_street": "street",
"addr_housenumber": "house_number",
"addr_postcode": "postal_code",
"addr_city": "city",
"contact_phone": "phone",
"contact_email": "email"
}


bakeries = bakeries_raw.rename(columns=rename_map)

# 4. Exclude Overlapping POI Categories

To avoid duplication with other POI layers (Cafes, Supermarkets), we exclude bakeries primarily tagged as:

- cafe

- restaurant

- fast_food

In [42]:
excluded_amenities = ["cafe", "restaurant", "fast_food", "supermarket"]


if "amenity" in bakeries.columns:
 bakeries = bakeries[~bakeries["amenity"].isin(excluded_amenities)]


len(bakeries)

1310

#  DATA TRANSFORMATION & ENRICHMENT

In [43]:
#  Unified Field Creation & Enrichment ---
# Create full address string
bakeries['address'] = bakeries.apply(
    lambda x: f"{x.get('street', '')} {x.get('house_number', '')}, {x.get('postal_code', '')} Berlin".strip(", "), 
    axis=1
)

In [44]:
# Sunday Opening Extraction

def get_sunday_opening(hours):
    # 1. If opening_hours is missing, we DON'T KNOW -> NaN
    if pd.isna(hours) or str(hours).strip() == "" or str(hours).lower() == 'nan':
        return np.nan
    
    h = str(hours).lower()
    
    # 2. Check if Sunday is explicitly mentioned as open
    if any(day in h for day in ["su", "sun", "0:00-24:00", "24/7"]):
        # Check if it says "Su off" (Explicitly closed)
        if "su off" in h or "sun off" in h:
            return False
        return True
    
    # 3. If hours exist but Sunday isn't mentioned: 
    # Returning NaN here is the "safest" way to avoid assumptions.
    return np.nan 

# Apply the function and keep it as 'object' type to preserve NaNs
bakeries['sunday_opening'] = bakeries['opening_hours'].apply(get_sunday_opening).astype(object)

In [45]:
# Phone Number Consolidation

def get_single_column(df, col_name):
    """Helper to ensure we only get one Series even if duplicate columns exist"""
    selected = df.get(col_name)
    if selected is None:
        return pd.Series([np.nan] * len(df), index=df.index)
    # If it's a DataFrame (multiple columns), take the first one
    if isinstance(selected, pd.DataFrame):
        return selected.iloc[:, 0]
    return selected

# 1. Safely extract single series for both potential tags
phone_series = get_single_column(bakeries, 'phone')
contact_series = get_single_column(bakeries, 'contact_phone')

# 2. Combine them (prioritize 'phone', fill with 'contact_phone')
bakeries['phone_combined'] = phone_series.fillna(contact_series)
# 3. Assign to the 'phone' column for the final selection
bakeries['phone'] = bakeries['phone_combined']


print(f"Success! Captured {bakeries['phone'].notna().sum()} phone numbers.")

Success! Captured phone    66
phone    66
dtype: int64 phone numbers.


# 5. Bakery Type & Chain Classification
## Logic

 - Presence of brand or operator → chain bakery

 - Otherwise → artisanal bakery

In [46]:
def classify_bakery(row):
 if pd.notna(row.get("brand")) or pd.notna(row.get("operator")):
  return "chain"
 return "artisanal"


bakeries["bakery_type"] = bakeries.apply(classify_bakery, axis=1)
bakeries["is_chain"] = bakeries["bakery_type"] == "chain"
bakeries["brand_name"] = bakeries["brand"].fillna(bakeries.get("operator", "Independent"))
bakeries["data_source"] = "OSM"

In [47]:
print(bakeries[["brand", "operator", "brand_name", "bakery_type", "is_chain"]].sample(10))

          brand      operator    brand_name bakery_type  is_chain
630   Steinecke           NaN     Steinecke       chain      True
1199        NaN  Fotografiska  Fotografiska       chain      True
249         NaN           NaN           NaN   artisanal     False
918         NaN      Krautzig      Krautzig       chain      True
95          NaN           NaN           NaN   artisanal     False
1156        NaN           NaN           NaN   artisanal     False
633   Steinecke           NaN     Steinecke       chain      True
380         NaN           NaN           NaN   artisanal     False
211         NaN           NaN           NaN   artisanal     False
1093        NaN           NaN           NaN   artisanal     False


# 6.Geospatial Validation 
 

## 6.1 CRS & Geometry Validity

In [48]:
bakeries = bakeries.set_crs(epsg=4326, allow_override=True)

# Remove rows with missing geometry
bakeries = bakeries[bakeries.geometry.notnull()]

# Remove invalid geometries
bakeries = bakeries[bakeries.geometry.is_valid]


#  Force a clean, warning-free coordinate extraction
# We project to meters (25833), get the center, then back to degrees (4326)
centroids = bakeries.geometry.to_crs(epsg=25833).centroid.to_crs(epsg=4326)
bakeries["latitude"] = centroids.y
bakeries["longitude"] = centroids.x

## 6.2 GEOPROCESSING

In [49]:
# Create GeoDataFrame
bakery_gdf = gpd.GeoDataFrame(
    bakeries,
    geometry=gpd.points_from_xy(bakeries.longitude, bakeries.latitude),
    crs="EPSG:4326"
)
bakery_gdf

,element,id,geometry,city,addr_country,house_number,postal_code,street,addr_suburb,shop,...,description_de,address,sunday_opening,phone_combined,bakery_type,is_chain,brand_name,data_source,latitude,longitude
0,node,58489996,POINT (13.40829 52.51035),Berlin,DE,77,10179,Alte Jakobstraße,Mitte,bakery,...,NaN,"Alte Jakobstraße 77, postal_code 10179\npos...",NaN,NaN,artisanal,False,NaN,OSM,52.510346,13.408290
1,node,253999302,POINT (13.35618 52.48628),Berlin,DE,22,10827,Hauptstraße,Schöneberg,bakery,...,NaN,"Hauptstraße 22, postal_code 10827\npostal_c...",True,NaN,chain,True,Back-Factory,OSM,52.486277,13.356180
2,node,254333065,POINT (13.32023 52.49613),NaN,NaN,NaN,NaN,NaN,NaN,bakery,...,NaN,"nan nan, postal_code NaN\npostal_code Na...",True,NaN,artisanal,False,NaN,OSM,52.496132,13.320234
3,node,254992626,POINT (13.26299 52.48796),NaN,NaN,NaN,NaN,NaN,NaN,bakery,...,NaN,"nan nan, postal_code NaN\npostal_code Na...",True,NaN,chain,True,Steinecke,OSM,52.487959,13.262985
4,node,258050855,POINT (13.29729 52.50696),Berlin,DE,80,10627,Kantstraße,Charlottenburg,bakery,...,NaN,"Kantstraße 80, postal_code 10627\npostal_co...",True,NaN,artisanal,False,NaN,OSM,52.506957,13.297293
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1363,way,1003281108,POINT (13.4345 52.51005),NaN,NaN,NaN,NaN,NaN,NaN,bakery,...,NaN,"nan nan, postal_code NaN\npostal_code Na...",True,NaN,chain,True,Steinecke,OSM,52.510051,13.434505
1364,way,1048012542,POINT (13.54639 52.43717),NaN,NaN,NaN,NaN,NaN,NaN,bakery,...,NaN,"nan nan, postal_code NaN\npostal_code Na...",NaN,NaN,artisanal,False,NaN,OSM,52.437165,13.546385
1365,way,1197629867,POINT (13.41497 52.51947),NaN,NaN,NaN,NaN,NaN,NaN,bakery,...,NaN,"nan nan, postal_code NaN\npostal_code Na...",NaN,NaN,chain,True,Kamps,OSM,52.519470,13.414973
1366,way,1410961516,POINT (13.34376 52.52648),NaN,NaN,NaN,NaN,Turmstraße,NaN,bakery,...,NaN,"Turmstraße nan, postal_code NaN\npostal_cod...",True,NaN,chain,True,Kamps,OSM,52.526478,13.343760


# 7. SPATIAL ENRICHMENT (DISTRICT & NEIGHBORHOOD)

Each bakery must be mapped to:

 - District

 - Neighborhood

 - District ID

 - Neighborhood ID

In [50]:
# Load official Berlin districts GeoDataFrame from lor_ortsteile.geojson
berlin_districts_gdf = gpd.read_file("../../mapping/lor_ortsteile.geojson")
berlin_districts_gdf = berlin_districts_gdf.to_crs(epsg=4326)


In [51]:
# Spatial join: matching your bakeries with district and neighborhoods
bakery_with_districts = gpd.sjoin(
    bakery_gdf,
    berlin_districts_gdf[["BEZIRK", "spatial_name", "OTEIL","geometry"]],
    how="left",
    predicate="within"
)
bakery_with_districts.head()

,element,id,geometry,city,addr_country,house_number,postal_code,street,addr_suburb,shop,...,bakery_type,is_chain,brand_name,data_source,latitude,longitude,index_right,BEZIRK,spatial_name,OTEIL
0,node,58489996,POINT (13.40829 52.51035),Berlin,DE,77,10179,Alte Jakobstraße,Mitte,bakery,...,artisanal,False,NaN,OSM,52.510346,13.408290,0,Mitte,0101,Mitte
1,node,253999302,POINT (13.35618 52.48628),Berlin,DE,22,10827,Hauptstraße,Schöneberg,bakery,...,chain,True,Back-Factory,OSM,52.486277,13.356180,44,Tempelhof-Schöneberg,0701,Schöneberg
2,node,254333065,POINT (13.32023 52.49613),NaN,NaN,NaN,NaN,NaN,NaN,bakery,...,artisanal,False,NaN,OSM,52.496132,13.320234,22,Charlottenburg-Wilmersdorf,0402,Wilmersdorf
3,node,254992626,POINT (13.26299 52.48796),NaN,NaN,NaN,NaN,NaN,NaN,bakery,...,chain,True,Steinecke,OSM,52.487959,13.262985,24,Charlottenburg-Wilmersdorf,0404,Grunewald
4,node,258050855,POINT (13.29729 52.50696),Berlin,DE,80,10627,Kantstraße,Charlottenburg,bakery,...,artisanal,False,NaN,OSM,52.506957,13.297293,21,Charlottenburg-Wilmersdorf,0401,Charlottenburg


In [52]:
print("latitude" in bakery_with_districts.columns)

True


In [53]:
print(bakery_with_districts["latitude"].isnull().sum())

0


In [54]:
## Renaming columns for proper schema
bakery_with_districts = bakery_with_districts.rename(columns={
    "BEZIRK": "district",
    "OTEIL": "neighborhood",
    "spatial_name": "neighborhood_id"
}).drop(columns=["index_right"])  # drop district_number if not needed
bakery_with_districts.head()

,element,id,geometry,city,addr_country,house_number,postal_code,street,addr_suburb,shop,...,phone_combined,bakery_type,is_chain,brand_name,data_source,latitude,longitude,district,neighborhood_id,neighborhood
0,node,58489996,POINT (13.40829 52.51035),Berlin,DE,77,10179,Alte Jakobstraße,Mitte,bakery,...,NaN,artisanal,False,NaN,OSM,52.510346,13.408290,Mitte,0101,Mitte
1,node,253999302,POINT (13.35618 52.48628),Berlin,DE,22,10827,Hauptstraße,Schöneberg,bakery,...,NaN,chain,True,Back-Factory,OSM,52.486277,13.356180,Tempelhof-Schöneberg,0701,Schöneberg
2,node,254333065,POINT (13.32023 52.49613),NaN,NaN,NaN,NaN,NaN,NaN,bakery,...,NaN,artisanal,False,NaN,OSM,52.496132,13.320234,Charlottenburg-Wilmersdorf,0402,Wilmersdorf
3,node,254992626,POINT (13.26299 52.48796),NaN,NaN,NaN,NaN,NaN,NaN,bakery,...,NaN,chain,True,Steinecke,OSM,52.487959,13.262985,Charlottenburg-Wilmersdorf,0404,Grunewald
4,node,258050855,POINT (13.29729 52.50696),Berlin,DE,80,10627,Kantstraße,Charlottenburg,bakery,...,NaN,artisanal,False,NaN,OSM,52.506957,13.297293,Charlottenburg-Wilmersdorf,0401,Charlottenburg


In [55]:
# District mapping (official codes as strings)
district_mapping = {
    'Mitte': '11001001',
    'Friedrichshain-Kreuzberg': '11002002',
    'Pankow': '11003003',
    'Charlottenburg-Wilmersdorf': '11004004',
    'Spandau': '11005005',
    'Steglitz-Zehlendorf': '11006006',
    'Tempelhof-Schöneberg': '11007007',
    'Neukölln': '11008008',
    'Treptow-Köpenick': '11009009',
    'Marzahn-Hellersdorf': '11010010',
    'Lichtenberg': '11011011',
    'Reinickendorf': '11012012'
}

# Apply mapping to create district_id column (string)
bakery_with_districts['district_id'] = bakery_with_districts['district'].map(district_mapping).astype(str)

# (Optional) Check if some districts were not mapped
unmapped = bakery_with_districts[~bakery_with_districts['district'].isin(district_mapping.keys())]['district'].unique()
if len(unmapped) > 0:
    print(" Unmapped districts found:", unmapped)
bakery_with_districts.head()

,element,id,geometry,city,addr_country,house_number,postal_code,street,addr_suburb,shop,...,bakery_type,is_chain,brand_name,data_source,latitude,longitude,district,neighborhood_id,neighborhood,district_id
0,node,58489996,POINT (13.40829 52.51035),Berlin,DE,77,10179,Alte Jakobstraße,Mitte,bakery,...,artisanal,False,NaN,OSM,52.510346,13.408290,Mitte,0101,Mitte,11001001
1,node,253999302,POINT (13.35618 52.48628),Berlin,DE,22,10827,Hauptstraße,Schöneberg,bakery,...,chain,True,Back-Factory,OSM,52.486277,13.356180,Tempelhof-Schöneberg,0701,Schöneberg,11007007
2,node,254333065,POINT (13.32023 52.49613),NaN,NaN,NaN,NaN,NaN,NaN,bakery,...,artisanal,False,NaN,OSM,52.496132,13.320234,Charlottenburg-Wilmersdorf,0402,Wilmersdorf,11004004
3,node,254992626,POINT (13.26299 52.48796),NaN,NaN,NaN,NaN,NaN,NaN,bakery,...,chain,True,Steinecke,OSM,52.487959,13.262985,Charlottenburg-Wilmersdorf,0404,Grunewald,11004004
4,node,258050855,POINT (13.29729 52.50696),Berlin,DE,80,10627,Kantstraße,Charlottenburg,bakery,...,artisanal,False,NaN,OSM,52.506957,13.297293,Charlottenburg-Wilmersdorf,0401,Charlottenburg,11004004


In [56]:
# View relevant columns
bakery_with_districts[
    ["name", "district", "district_id", "neighborhood", "neighborhood_id"]
].head()

,name,district,district_id,neighborhood,neighborhood_id
0,NaN,Mitte,11001001,Mitte,0101
1,Back-Factory,Tempelhof-Schöneberg,11007007,Schöneberg,0701
2,Brot & Brötchen,Charlottenburg-Wilmersdorf,11004004,Wilmersdorf,0402
3,Steinecke,Charlottenburg-Wilmersdorf,11004004,Grunewald,0404
4,Wilmina Brot,Charlottenburg-Wilmersdorf,11004004,Charlottenburg,0401


In [57]:
# View columns related to OSM tags
[col for col in bakery_with_districts.columns if "osm" in col.lower()]
bakery_with_districts.columns


Index(['element', 'id', 'geometry', 'city', 'addr_country', 'house_number',
       'postal_code', 'street', 'addr_suburb', 'shop',
       ...
       'bakery_type', 'is_chain', 'brand_name', 'data_source', 'latitude',
       'longitude', 'district', 'neighborhood_id', 'neighborhood',
       'district_id'],
      dtype='object', length=217)

#  Geometry Standardization

In [58]:
#  1. Perform the check : the distribution of geometry types
if 'geometry' in bakery_with_districts.columns:
    print("Geometry Type Counts:")
    print(bakery_with_districts.geometry.geom_type.value_counts())

    # Check if there are any non-point geometries
    non_points = bakery_with_districts[bakery_with_districts.geometry.geom_type != 'Point']
    if not non_points.empty:
        print(f"\n Found {len(non_points)} records that are NOT Points.")
        print("These will be converted to centroids/representative points.")
    else:
        print("\n All records are already Points.")
else:
    print("No 'geometry' column found. Ensure you have converted lat/long to geometry first.")

Geometry Type Counts:
Point    1310
Name: count, dtype: int64

 All records are already Points.


In [59]:
# 2. Apply Defensive Transformation

# Even if 100% are currently Points, this keeps the code robust for future data
bakery_with_districts['geometry'] = bakery_with_districts.geometry.apply(lambda g: g if g.geom_type == 'Point' else g.representative_point())
# 3. Final Verification
print("Confirmed geometry types:", bakery_with_districts.geometry.geom_type.unique())


Confirmed geometry types: ['Point']


## Identity Linkage: Bakery vs. Supermarket Cross-Check

In [60]:
import os
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

# 1. Load the .env file from the local directory
load_dotenv()

# 2. Retrieve credentials from environment variables
user = os.getenv('DB_USER')
password = os.getenv('DB_PASSWORD')
host = os.getenv('DB_HOST')
port = os.getenv('DB_PORT')
db = os.getenv('DB_NAME')
ssl = os.getenv('DB_SSLMODE', 'require') # Defaults to 'require' if not in .env

# 3. Construct the connection string
# Using the specific format required for your Neon/PostgreSQL database
DATABASE_URL = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{db}?sslmode={ssl}"

# 4. Create the engine
engine = create_engine(DATABASE_URL)

print(" Environment variables loaded and engine created.")

 Environment variables loaded and engine created.


In [61]:
# Add this right before your 'with engine.connect()' line
print(f"DEBUG: Connecting to {engine.url.host} on database {engine.url.database}")

DEBUG: Connecting to localhost on database layereddb


In [62]:

# 2. Query only the store_id column from the supermarket table
query = "SELECT id FROM berlin_source_data.supermarkets"

with engine.connect() as conn:
    # Load supermarket IDs from the database
    db_supermarket_ids = pd.read_sql(text(query), conn)['id']

# 3. Convert both to the same type (strings) to ensure a fair comparison
# Assuming  bakery IDs are already cleaned/numeric strings
bakery_ids = set(bakery_with_districts['id'].astype(str))
supermarket_ids = set(db_supermarket_ids.astype(str))

# 4. Calculate the intersection
overlap = bakery_ids.intersection(supermarket_ids)
overlap_count = len(overlap)

print(f"Total Bakeries: {len(bakery_ids)}")
print(f"Total Supermarkets in DB: {len(supermarket_ids)}")
print(f"---")
print(f"Number of IDs found in both: {overlap_count}")

if overlap_count > 0:
    print(f"Sample overlapping IDs: {list(overlap)[:5]}")
else:
    print(" No overlapping IDs found. Data is unique!")

Total Bakeries: 1310
Total Supermarkets in DB: 1355
---
Number of IDs found in both: 0
 No overlapping IDs found. Data is unique!


# 8. Final Unified Schema Selection

In [63]:
final_columns = [
"id",
"name",
"bakery_type",
"is_chain",
"brand_name",
"address",
"street",
"house_number",
"postal_code",
"phone",
"amenities",
"accessibility",
"opening_hours",
"sunday_opening",
"website",
"latitude",
"longitude",
"geometry",
"district",
"neighborhood",
"district_id",
"neighborhood_id",
"data_source"

]
# Ensure all mandatory columns exist (fill with None if missing)
for col in final_columns:
    if col not in bakery_with_districts.columns:
        bakery_with_districts[col] = None
        
final_bakeries = bakery_with_districts[final_columns].copy()
final_bakeries.head()

,id,name,bakery_type,is_chain,brand_name,address,street,house_number,postal_code,postal_code,...,sunday_opening,website,latitude,longitude,geometry,district,neighborhood,district_id,neighborhood_id,data_source
0,58489996,NaN,artisanal,False,NaN,"Alte Jakobstraße 77, postal_code 10179\npos...",Alte Jakobstraße,77,10179,NaN,...,NaN,NaN,52.510346,13.408290,POINT (13.40829 52.51035),Mitte,Mitte,11001001,0101,OSM
1,253999302,Back-Factory,chain,True,Back-Factory,"Hauptstraße 22, postal_code 10827\npostal_c...",Hauptstraße,22,10827,NaN,...,True,NaN,52.486277,13.356180,POINT (13.35618 52.48628),Tempelhof-Schöneberg,Schöneberg,11007007,0701,OSM
2,254333065,Brot & Brötchen,artisanal,False,NaN,"nan nan, postal_code NaN\npostal_code Na...",NaN,NaN,NaN,NaN,...,True,NaN,52.496132,13.320234,POINT (13.32023 52.49613),Charlottenburg-Wilmersdorf,Wilmersdorf,11004004,0402,OSM
3,254992626,Steinecke,chain,True,Steinecke,"nan nan, postal_code NaN\npostal_code Na...",NaN,NaN,NaN,NaN,...,True,NaN,52.487959,13.262985,POINT (13.26299 52.48796),Charlottenburg-Wilmersdorf,Grunewald,11004004,0404,OSM
4,258050855,Wilmina Brot,artisanal,False,NaN,"Kantstraße 80, postal_code 10627\npostal_co...",Kantstraße,80,10627,NaN,...,True,NaN,52.506957,13.297293,POINT (13.29729 52.50696),Charlottenburg-Wilmersdorf,Charlottenburg,11004004,0401,OSM


# 9. Quality Assurance Checks

In [64]:
qa = {
"total_records": len(final_bakeries),
"missing_names": final_bakeries["name"].isna().sum(),
"missing_coords": final_bakeries[
final_bakeries["latitude"].isna() | final_bakeries["longitude"].isna()
].shape[0] if 'latitude' in final_bakeries.columns else len(final_bakeries) ,
# Handle the mean calculation carefully if is_chain is all None
    "chain_share_percent": round(pd.to_numeric(final_bakeries["is_chain"], errors='coerce').mean() * 100, 2) 
                           if final_bakeries["is_chain"].notna().any() else 0.0
}


qa

{'total_records': 1310,
 'missing_names': np.int64(21),
 'missing_coords': 0,
 'chain_share_percent': np.float64(25.04)}

In [65]:
print(final_bakeries["latitude"].head())
print(final_bakeries["longitude"].head())

0    52.510346
1    52.486277
2    52.496132
3    52.487959
4    52.506957
Name: latitude, dtype: float64
0    13.408290
1    13.356180
2    13.320234
3    13.262985
4    13.297293
Name: longitude, dtype: float64


In [66]:
# Returns a list of the names that are duplicates
duplicate_column_names = final_bakeries.columns[final_bakeries.columns.duplicated()].tolist()

if duplicate_column_names:
    print(f" Duplicate columns found: {duplicate_column_names}")
else:
    print(" No duplicate column names found.")

 Duplicate columns found: ['postal_code', 'phone']


In [67]:
# 1. Fix Duplicate Columns (Keeping the one with more data)
# This identifies columns with the same name and drops the one that is mostly empty
cols = pd.Series(final_bakeries.columns)
for dup in cols[cols.duplicated()].unique():
    # Keep the first, drop the rest
    cols_to_drop = [i for i, v in enumerate(final_bakeries.columns) if v == dup][1:]
    final_bakeries = final_bakeries.iloc[:, ~final_bakeries.columns.duplicated()]

In [68]:
# Fix the 21 missing bakery names
final_bakeries['name'] = final_bakeries['name'].fillna("Unknown Bakery")

In [69]:
# Map original tags to the unified 'amenities' column
def map_amenities(row):
    items = []
    if row.get('outdoor_seating') == 'yes': items.append('outdoor_seating')
    if row.get('takeaway') == 'yes': items.append('coffee_to_go')
    if row.get('delivery') == 'yes': items.append('delivery')
    # Return as string or 'None' if empty
    return ", ".join(items) if items else "Standard Bakery"

# Map wheelchair tag to 'accessibility'
def map_accessibility(row):
    wc = str(row.get('wheelchair', 'unknown')).lower()
    if wc == 'yes': return 'Fully Accessible'
    if wc == 'limited': return 'Partial Access'
    return 'Not Specified'

# Apply to the dataframe that still has the raw columns (bakeries or bakery_with_districts)
final_bakeries['amenities'] = bakery_with_districts.apply(map_amenities, axis=1)
final_bakeries['accessibility'] = bakery_with_districts.apply(map_accessibility, axis=1)

In [70]:
# Convert any remaining NaN in sunday_opening to False (assuming closed if not specified)
#final_bakeries['sunday_opening'] = final_bakeries['sunday_opening'].fillna(False).astype(bool)

In [71]:
# This shows you how many are Open, Closed, and Unknown
print(final_bakeries['sunday_opening'].value_counts(dropna=False))

sunday_opening
True     654
NaN      629
False     27
Name: count, dtype: int64


In [72]:
final_bakeries.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 1310 entries, 0 to 1367
Data columns (total 23 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   id               1310 non-null   int64   
 1   name             1310 non-null   object  
 2   bakery_type      1310 non-null   object  
 3   is_chain         1310 non-null   bool    
 4   brand_name       328 non-null    object  
 5   address          1310 non-null   object  
 6   street           717 non-null    object  
 7   house_number     691 non-null    object  
 8   postal_code      660 non-null    object  
 9   phone            66 non-null     object  
 10  amenities        1310 non-null   object  
 11  accessibility    1310 non-null   object  
 12  opening_hours    922 non-null    object  
 13  sunday_opening   681 non-null    object  
 14  website          191 non-null    object  
 15  latitude         1310 non-null   float64 
 16  longitude        1310 non-null   float6

In [73]:

#  Now check the ' phone' count 
phone_count = final_bakeries['phone'].notna().sum()
print(f"Success! Captured {phone_count} unique phone numbers.")

# 3. Double-check that 'phone' is now a single Series, not a DataFrame
print(f"Is 'phone' a single column? {isinstance(final_bakeries['phone'], pd.Series)}")

Success! Captured 66 unique phone numbers.
Is 'phone' a single column? True


In [74]:
print(" Dataset after Steps A - D cleaning and transforming\n")

# Shape of dataframe
print(f"Number of rows: {final_bakeries.shape[0]}")
print(f"Number of columns: {final_bakeries.shape[1]}")

# Column list
print("\nRemaining columns:")
print(final_bakeries.columns.tolist())

# Missing values check
missing = final_bakeries.isnull().sum()
print("\nMissing values after cleaning and transforming :")
print(missing)

 Dataset after Steps A - D cleaning and transforming

Number of rows: 1310
Number of columns: 23

Remaining columns:
['id', 'name', 'bakery_type', 'is_chain', 'brand_name', 'address', 'street', 'house_number', 'postal_code', 'phone', 'amenities', 'accessibility', 'opening_hours', 'sunday_opening', 'website', 'latitude', 'longitude', 'geometry', 'district', 'neighborhood', 'district_id', 'neighborhood_id', 'data_source']

Missing values after cleaning and transforming :
id                    0
name                  0
bakery_type           0
is_chain              0
brand_name          982
address               0
street              593
house_number        619
postal_code         650
phone              1244
amenities             0
accessibility         0
opening_hours       388
sunday_opening      629
website            1119
latitude              0
longitude             0
geometry              0
district              0
neighborhood          0
district_id           0
neighborhood_id       

# 11. Save Processed Output

In [75]:

# Save final bakeries dataset as CSV (non-spatial version)
final_bakeries.drop(columns=["geometry"], errors="ignore").to_csv(
    "../sources/bakeries_berlin.csv",
    index=False
)
# Save final bakeries dataset as GeoJSON (spatial version)
final_bakeries.to_file(
    "../sources/bakeries_berlin.geojson",
    driver="GeoJSON"
)

In [76]:
try:
    with engine.connect() as conn:
        print("Connection is still active and ready for population.")
except NameError:
    print("Engine not found. Please re-run your connection setup cell.")

Connection is still active and ready for population.


# Populating Bakeries Data to DB

## Step 1: Initialize Database Connection & Schema

In [77]:
# --- STEP 1: CREATE THE TABLE ---

# The SQL statement to create the bakeries table with all required constraints
create_table_sql = """
CREATE TABLE IF NOT EXISTS berlin_source_data.bakeries (
    id VARCHAR(20) PRIMARY KEY,
    name VARCHAR(200) NOT NULL,
    latitude DECIMAL(9,6),
    longitude DECIMAL(9,6),
    geometry VARCHAR,
    neighborhood VARCHAR(100),
    district VARCHAR(100),
    district_id VARCHAR(20) NOT NULL,
    neighborhood_id VARCHAR(20),
    opening_hours VARCHAR(200),
    website VARCHAR(200),
    phone_number VARCHAR(50),
    organic BOOLEAN DEFAULT FALSE,
    CONSTRAINT district_id_fk FOREIGN KEY (district_id) 
        REFERENCES berlin_source_data.districts(district_id) 
        ON DELETE RESTRICT 
        ON UPDATE CASCADE
);
"""

# Execute the creation using your existing 'engine'
with engine.connect() as conn:
    conn.execute(text(create_table_sql))
    conn.commit()
    
print("Step 1 Complete: Table 'berlin_source_data.bakeries' is ready.")

Step 1 Complete: Table 'berlin_source_data.bakeries' is ready.


## Step 2: Prepare Data for Injection

In [78]:
# --- STEP 2: FORMAT DATA ---

# 1. Mapping final dataset columns to the database column names
db_mapping = {
    'id': 'id',
    'name': 'name',
    'latitude': 'latitude',
    'longitude': 'longitude',
    'opening_hours': 'opening_hours',
    'phone': 'phone_number',
    'website': 'website',
    'district': 'district',
    'district_id': 'district_id',
    'neighborhood': 'neighborhood',
    'neighborhood_id': 'neighborhood_id'
}

# 2. Rename and filter the dataframe for database upload
#  Convert to a standard Pandas DataFrame to avoid GeoPandas warnings
# We use .copy() to ensure we don't modify the original final_bakeries

to_db_df = pd.DataFrame(final_bakeries.rename(columns=db_mapping)).copy()

# 3. Convert geometry to WKT(Well-Known Text) string format for the VARCHAR column
to_db_df['geometry'] = to_db_df['geometry'].apply(lambda x: x.wkt if hasattr(x, 'wkt') else str(x))


# 4. Select only the columns that exist in the database table
db_columns = [
    'id',  'name', 'latitude', 'longitude', 
    'geometry', 'neighborhood', 'district', 'district_id', 'neighborhood_id',
    'opening_hours', 'website', 'phone_number'
]

final_payload = to_db_df[db_columns]

print(f"Step 2 Complete: {len(final_payload)} records prepared for upload.")
#print(f"Columns ready for upload: {list(final_payload.columns)}")

Step 2 Complete: 1310 records prepared for upload.


## Step 3: Database Population and Final Verification

In [79]:
# --- STEP 3: DATA UPLOAD & AUDIT ---

try:
    # 1. Execute the Data Load
    # We use 'append' as the table was established in Step 1.
    # We use the final_payload prepared in  previous step.
    final_payload.to_sql(
        name='bakeries', 
        con=engine, 
        schema='berlin_source_data', 
        if_exists='append', 
        index=False,
        method='multi'
    )
    
    print(f" Success! {len(final_payload)} records successfully uploaded to 'berlin_source_data.bakeries'.")

    # 2. Final Verification Audit
    # Querying the database directly to confirm counts match 'final_bakeries' data.
    with engine.connect() as conn:
        # Total row count
        result = conn.execute(text("SELECT COUNT(*) FROM berlin_source_data.bakeries"))
        db_count = result.scalar()
        
        print(f"\n--- Database Audit Report ---")
        print(f"Total records in DB: {db_count}")
        
        if db_count == len(final_payload):
            print(f" Data integrity verified: DB count matches local dataframe.")
        else:
            print(f" Warning: DB count ({db_count}) differs from local count ({len(final_payload)}).")

except Exception as e:
    print(f" Error during database upload: {e}")
    print("\n--- Quick Troubleshooting ---")
    if "foreign key" in str(e).lower():
        print(" Check: Do all 'district_id' values exist in the districts parent table?")
    elif "unique constraint" in str(e).lower():
        print(" Check: There is a duplicate 'id' in your data. Ensure you ran drop_duplicates().")

Success! 1310 records successfully uploaded to 'berlin_source_data.bakeries'.

--- Database Audit Report ---
Total records in DB: 1310
 Data integrity verified: DB count matches local dataframe.


In [80]:
# fetch few rows from the bakeries table to verify data upload
with engine.connect() as conn:
    result = conn.execute(text("SELECT * FROM berlin_source_data.bakeries LIMIT 5"))
    rows = result.fetchall()
    for row in rows:
        print(row)

('58489996', 'Unknown Bakery', Decimal('52.510346'), Decimal('13.408290'), 'POINT (13.4082896 52.510345699999995)', 'Mitte', 'Mitte', '11001001', '0101', None, None, None, False)
('253999302', 'Back-Factory', Decimal('52.486277'), Decimal('13.356180'), 'POINT (13.3561798 52.4862765)', 'Schöneberg', 'Tempelhof-Schöneberg', '11007007', '0701', 'Mo-Fr 06:00-19:00; Sa 06:30-18:00; Su 07:00-16:00', None, None, False)
('254333065', 'Brot & Brötchen', Decimal('52.496132'), Decimal('13.320234'), 'POINT (13.3202337 52.4961316)', 'Wilmersdorf', 'Charlottenburg-Wilmersdorf', '11004004', '0402', 'Mo-Fr 07:00-17:00; Sa 07:00-14:00; Su 07:00-14:00', None, None, False)
('254992626', 'Steinecke', Decimal('52.487959'), Decimal('13.262985'), 'POINT (13.2629853 52.487958899999995)', 'Grunewald', 'Charlottenburg-Wilmersdorf', '11004004', '0404', 'Mo-Fr 06:30-18:00; Sa 07:00-14:00; Su 07:00-12:00', None, None, False)
('258050855', 'Wilmina Brot', Decimal('52.506957'), Decimal('13.297293'), 'POINT (13.29729